## Train the PF-Controller Neural Network

For each performance metric, train the corresponding particle-filter (PF) controller neural network.

In [ ]:
import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt
import torch.nn as nn
from src.models.particle_filter.core import ParticleFilter
from src.models.networks.pf_mlp import ParticleFilterMLP
from src.helpers.seed import set_global_seed

from experiment_config import (
    DegModel,DATA_NAME,SEED,LEAKY_SLOPE,HIDDEN_DIMS,LOSS_TAIL_SIZE,N_PARTICLES,PFNET_DIR,DEGR_MODEL_DIR,ESTIMATION_DIR,LOSS_WINDOW,AVERSION,DROPOUT,MAX_LIFE
)
    
DATA_NAME

## Task

In [ ]:
perform_name = 'T48'

## Hyper-Parameters

In [ ]:
# Training parameters#
# optimization
lr = 5e-4

# Early stopping
max_epochs = 300
min_epochs = 100
patience = 40     # how many eval steps to wait
min_delta = 0.01      # minimum improvement to count

In [ ]:
activation = nn.LeakyReLU(LEAKY_SLOPE)

set_global_seed(SEED)
PERFORM_DIR = PFNET_DIR/perform_name

checkpoint_best_path = PERFORM_DIR / "checkpoint_best.pt"
checkpoint_last_path = PERFORM_DIR / "checkpoint_last.pt"
checkpoint_best_path.parent.mkdir(parents=True, exist_ok=True)

## Import training data

In [ ]:
hi_df = pd.read_csv(ESTIMATION_DIR/"data_dev.csv")
units = hi_df['unit'].astype(int).unique().tolist()
units 

### Extract degradation onset

In [ ]:
onsets = {unit: hi_df[(hi_df['unit']==unit) & (hi_df['hs']==0)]['cycle'].values[0] for unit in units}
del hi_df['hs']

In [ ]:
perform_names = [col for col in hi_df.columns if col not in ['unit','cycle']]

performs = {name: 
    {unit: hi_df[hi_df['unit']==unit][name].values for unit in units} 
    for name in perform_names
}

init_ss = {name: {unit: perform[unit].max() for unit in units} for name,perform in performs.items()}
time = {unit: hi_df[hi_df['unit']==unit]['cycle'].values for unit in units}

## Create component (base) models

In [ ]:
dev_data = {}
dev_eol = {}
for unit in units:
    t = time[unit]
    s = performs[perform_name][unit]
    dev_data[unit]=torch.tensor(np.stack([t, s],axis=1),dtype=torch.float32)
    dev_eol[unit]=torch.tensor(t[-1],dtype=torch.float32) 


In [ ]:
dev_degmodels = {}
for unit, perform in performs[perform_name].items():
	best_model = DegModel(onset=onsets[unit], init_s=init_ss[perform_name][unit])
	best_model.load_state_dict(
		torch.load(DEGR_MODEL_DIR/'states'/perform_name/f'unit_{unit}'/ "best_model.pt")
	)
	dev_degmodels[unit] = best_model

## Train Particle Filter controller Net

In [ ]:
net = ParticleFilterMLP(state_dim=DegModel.state_dim(), hidden_dims=HIDDEN_DIMS,
                        activation=lambda : nn.LeakyReLU(LEAKY_SLOPE), dropout_p=DROPOUT)

optimizer = torch.optim.Adam(net.parameters(), lr)
optimizer.zero_grad()

In [ ]:
best_score = float("inf")
train_epoch_losses = []
eval_epoch_losses = []
scores = []
window_means = []
window_stds = []
wait = 0
start_epoch = 0

if checkpoint_last_path.exists():
    ckpt = torch.load(checkpoint_last_path, weights_only=False)

    net.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])

    start_epoch = ckpt["epoch"] + 1
    best_score = ckpt["best_score"]
    scores = ckpt.get("scores", [])
    train_epoch_losses = ckpt.get("train_losses", [])
    eval_epoch_losses = ckpt.get("eval_losses", [])
    window_means = ckpt.get("window_means", [])
    window_stds = ckpt.get("window_stds", [])

    torch.random.set_rng_state(ckpt["rng_torch"])
    np.random.set_state(ckpt["rng_numpy"])

    print(f"✓ Resumed from epoch {start_epoch}, best_score={best_score:.3f}")


In [ ]:
for epoch in range(start_epoch, max_epochs):

    eval_fold_losses = []
    train_fold_losses = []

    for eval_unit in np.random.permutation(units):
        train_unit_losses = []

        # --------------------------
        # TRAINING
        # --------------------------
        for train_unit in np.random.permutation(units):

            if train_unit == eval_unit:
                continue

            train_offline_units = [
                u for u in units if u not in (train_unit, eval_unit)
            ]

            train_offline_degmodels = [
                dev_degmodels[u] for u in train_offline_units
            ]

            train_t_data = dev_data[train_unit][:, 0]
            train_s_data = dev_data[train_unit][:, 1]

            optimizer.zero_grad()

            train_pf = ParticleFilter(
                base_models=train_offline_degmodels,
                net=net,
                n_particles=N_PARTICLES,
                max_life = MAX_LIFE,
            ).train()
            
            train_step_losses = []
            for k in range(len(train_t_data)):
                train_mixture_dist = train_pf.step(
                    t_obs=train_t_data[[k]],
                    s_obs=train_s_data[[k]],
                )

                start_loss = -LOSS_TAIL_SIZE if LOSS_TAIL_SIZE else k
                train_last_dist = train_mixture_dist.distribution(
                    s=train_s_data[start_loss:]
                )
                train_step_nll = -train_last_dist.log_prob(
                    train_t_data[start_loss:]
                ).mean()

                train_step_losses.append(train_step_nll)

            train_unit_loss = torch.stack(train_step_losses).mean()

            train_unit_loss.backward()

            optimizer.step()

            train_unit_losses.append(train_unit_loss.item())

        train_fold_loss = np.mean(train_unit_losses)
        train_fold_losses.append(train_fold_loss)

        # --------------------------
        # EVALUATION
        # --------------------------

        eval_offline_units = [u for u in units if u != eval_unit]

        eval_offline_degmodels = [
            dev_degmodels[u] for u in eval_offline_units
        ]

        eval_t_data = dev_data[eval_unit][:, 0]
        eval_s_data = dev_data[eval_unit][:, 1]

        eval_pf = ParticleFilter(
            base_models=eval_offline_degmodels,
            net=net,
            n_particles=N_PARTICLES,
            max_life=MAX_LIFE
        ).eval()

        eval_step_losses = []
        for k in range(len(eval_t_data)):

            eval_mixture_dist = eval_pf.step(
                t_obs=eval_t_data[[k]],
                s_obs=eval_s_data[[k]],
            )

            start_loss = -LOSS_TAIL_SIZE if LOSS_TAIL_SIZE else k

            eval_last_dist = eval_mixture_dist.distribution(
                s=eval_s_data[start_loss:]
            )

            eval_step_nll = -eval_last_dist.log_prob(
                eval_t_data[start_loss:]
            ).mean()

            eval_step_losses.append(eval_step_nll.item())

        eval_fold_loss = np.mean(eval_step_losses)
        eval_fold_losses.append(eval_fold_loss)

    # --------------------------
    # epoch statistics
    # --------------------------

    train_epoch_loss = np.mean(train_fold_losses)
    eval_epoch_loss = np.mean(eval_fold_losses)

    train_epoch_losses.append(train_epoch_loss)
    eval_epoch_losses.append(eval_epoch_loss)



    # --------------------------
    # VARIANCE-AWARE SELECTION
    # --------------------------


    window = eval_epoch_losses[-LOSS_WINDOW:]
    mean_loss = np.mean(window)
    std_loss = np.std(window)
    window_means.append(mean_loss)
    window_stds.append(std_loss)

    score = mean_loss + AVERSION * std_loss
    scores.append(score)
    
    print(
        f"[Epoch {epoch:03d}] "
        f"train={train_epoch_loss:.3f} | "
        f"eval={eval_epoch_loss:.3f} | "
        f"score={score:.3f} "
        f"(μ={mean_loss:.3f}, σ={std_loss:.3f})"
    )


    # --------------------------
    # CHECKPOINT SELECTION
    # --------------------------

    if (epoch + 1 >= LOSS_WINDOW): 
        if (score < best_score - min_delta):

            best_score = score
            wait = 0

            best_checkpoint = {
                "epoch": epoch,
                "model_state": net.state_dict(),
                "best_score": best_score,
            }

            torch.save(best_checkpoint, checkpoint_best_path)

            last_checkpoint = {
                "epoch": epoch,
                "model_state": net.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_score": best_score,
                "scores": scores,
                "train_losses": train_epoch_losses,
                "eval_losses": eval_epoch_losses,
                "window_means": window_means,
                "window_stds": window_stds,
                "rng_torch": torch.random.get_rng_state(),
                "rng_numpy": np.random.get_state(),
            }

            torch.save(last_checkpoint, checkpoint_last_path)

            print(f"  + saved (score={score:.3f})")
        else:
            wait += 1

            if epoch >= max(min_epochs, LOSS_WINDOW) and wait >= patience:
                print("🛑 Early stopping triggered")
                break


# Save last results

In [ ]:
if start_epoch < max_epochs:
    last_checkpoint = {
        "epoch": epoch,
        "model_state": net.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_score": best_score,
        "scores": scores,
        "train_losses": train_epoch_losses,
        "eval_losses": eval_epoch_losses,
        "window_means": window_means,
        "window_stds": window_stds,
        "rng_torch": torch.random.get_rng_state(),
        "rng_numpy": np.random.get_state(),
    }

    torch.save(last_checkpoint, checkpoint_last_path)

    print("✓ Final LAST checkpoint saved.")

else:
    epoch = start_epoch

## Plot

In [ ]:
train_loss_arr = np.array(train_epoch_losses)
eval_loss_arr = np.array(eval_epoch_losses)
score_arr = np.array(scores)
best_epoch = best_checkpoint['epoch']

epochs = np.arange(len(train_loss_arr))

# best epoch based on score

best_score = score_arr[best_epoch]
best_eval_loss = eval_loss_arr[best_epoch]

plt.figure(figsize=(8, 5))

# Main curves
plt.plot(epochs, train_loss_arr, linewidth=2, label="train loss")
plt.plot(epochs, eval_loss_arr, linewidth=2, label="eval loss")
plt.plot(epochs, score_arr, linewidth=2, linestyle="--", label="score")

ymin, ymax = plt.ylim()
yrange = ymax - ymin

xmin, xmax = plt.xlim()
xrange = xmax - xmin

y_offset = 0.5 * yrange
x_offset = -0.05 * xrange

# Annotation of best score
plt.annotate(
    f"min score = {best_score:.3f}\nepoch = {best_epoch}",
    xy=(best_epoch, best_score),
    xytext=(best_epoch + x_offset, best_score + y_offset),
    textcoords="data",
    arrowprops=dict(arrowstyle="->", linewidth=1),
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8),
)

plt.xlabel("Epoch")
plt.ylabel("Loss / Score")
plt.title(f"{perform_name} PF network training")

plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

# Save figure
fig_path = PERFORM_DIR / "net_training_loss.png"
plt.savefig(fig_path, dpi=300)

# Save raw arrays
np.save(PERFORM_DIR / "net_training.npy", train_loss_arr)
np.save(PERFORM_DIR / "net_eval.npy", eval_loss_arr)
np.save(PERFORM_DIR / "net_score.npy", score_arr)
np.save(PERFORM_DIR / "net_bestepoch.npy", best_epoch)

plt.show()

print(f"✓ Plot saved to {fig_path}")
print(f"✓ Best epoch: {best_epoch} (score={best_score:.4f})")